# RAG from Scratch to Production — Episode 2/20
## Architecture: one request, end to end

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JakkulaMahendhar/rag-from-scratch/blob/main/notebooks/02_architecture.ipynb)

In Episode 1 we built RAG in 20 lines: **chunk → retrieve → augment → generate**. Then it broke: the user typed **"WFH"**, the handbook said **"remote"**, and retrieval found nothing.

A production system is those same 4 steps with more stages around them. In this episode we follow **one question** through a miniature copy of the [production app](https://github.com/JakkulaMahendhar/rag-chatbot), stage by stage, and map each stage to the real file.

**In this episode:**
1. See the two flows every RAG system has: **ingestion** (upload) and **query** (ask).
2. Build a mini ingestion flow: upload returns at once, a background worker does the work.
3. Build the full query flow in plain Python: access check → query rewrite → query expansion → retrieval → rerank → context management → prompt → generate → hallucination guard → sources.
4. Watch the Episode 1 **"WFH" question get answered**, and see what it costs.

Every stage here is a simple stand-in. Later episodes swap each one for the real thing.

## 0. Setup: pick an LLM

Same as Episode 1: one `generate(prompt) -> str` function, two providers.

- **Gemini** (easiest in Colab): get a free key at https://aistudio.google.com/apikey and add it as a Colab secret named `GEMINI_API_KEY`.
- **Ollama** (fully local): `ollama pull llama3.1`, then set `PROVIDER = "ollama"`.

In [ ]:
%pip install -q google-generativeai requests

In [1]:
import os, re, json, time, requests

PROVIDER = os.getenv("LLM_PROVIDER", "gemini")   # "gemini" or "ollama"
GEMINI_MODEL = "gemini-3.6-flash"
OLLAMA_MODEL = "llama3.1"


def _gemini_key():
    try:
        from google.colab import userdata  # Colab secrets
        return userdata.get("GEMINI_API_KEY")
    except Exception:
        return os.getenv("GEMINI_API_KEY") or __import__("getpass").getpass("GEMINI_API_KEY: ")


if PROVIDER == "gemini":
    import google.generativeai as genai
    genai.configure(api_key=_gemini_key())
    _model = genai.GenerativeModel(GEMINI_MODEL)


def generate(prompt: str) -> str:
    """One interface, two providers. Temperature 0 so results are repeatable."""
    if PROVIDER == "gemini":
        return _model.generate_content(
            prompt, generation_config={"temperature": 0}
        ).text.strip()
    r = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": OLLAMA_MODEL, "prompt": prompt, "stream": False,
              "options": {"temperature": 0}},
        timeout=300,
    )
    r.raise_for_status()
    return r.json()["response"].strip()


LLM_CALLS = []   # (purpose, seconds) for every LLM call in the current request

def llm(prompt: str, purpose: str) -> str:
    """generate(), plus a log entry so we can count what one request costs."""
    start = time.perf_counter()
    out = generate(prompt)
    LLM_CALLS.append((purpose, time.perf_counter() - start))
    return out


print(f"Using {PROVIDER}")

Using ollama


## 1. The map

Every RAG system has **two flows**. They run at different times, in different processes.

```
 1. INGESTION (once per document)                 2. QUERY (once per question)
 ────────────────────────────────                 ────────────────────────────
 POST /upload                                     POST /chat
   save file, row status="pending"                  1. access check (who is asking?)
   return 202 immediately                           2. rewrite the question   (LLM call)
        │                                           3. expand into 3 queries  (LLM call)
        ▼  background worker                        4. retrieve: hybrid search, top 10
   parse → chunk → embed                            5. rerank: top 10 → top 3
        │                                           6. dedupe, compress, fit the budget
        ▼                                           7. build prompt → answer  (LLM call)
   store in vector DB + BM25 index                  8. hallucination guard    (LLM call)
   status="completed"                               9. attach sources + quality score
```

| Stage | File in the [production app](https://github.com/JakkulaMahendhar/rag-chatbot) | Deep dive |
|---|---|---|
| Upload + worker | `app/api/upload.py`, `app/worker.py` | Ep 17 |
| Parse, chunk, embed | `parser.py`, `chunker.py`, `embedding.py` | Ep 3–5 |
| Access check | `query_access.py`, `auth/` | Ep 18 |
| Rewrite + expand | `query_enhancer.py`, `query_expander.py` | Ep 9 |
| Retrieve | `retrieval.py`, `hybrid_search.py` | Ep 6–8 |
| Rerank | `reranker.py` | Ep 10–11 |
| Context management | `context_deduplicator.py`, `context_compressor.py`, `context_window_manager.py` | Ep 12 |
| Prompt + generate | `prompt_builder.py`, `llm/` | Ep 13–14 |
| Hallucination guard | `hallucination_guard_service.py` | Ep 15 |
| Quality + sources | `search_evaluator.py`, `source_builder.py` | Ep 16 |

The orchestrator that runs the query flow is `app/services/rag_chat.py`. We will write a mini version of it below.

## 2. The documents: two users, two files

The same handbook as Episode 1, owned by **user 1** (an employee). Plus a salary file owned by **user 2** (HR). User 1 must never see it.

In [2]:
HANDBOOK = """
# NovaTech Solutions — Employee Handbook (2026 edition)

## Remote Work
Employees may work remotely up to 2 days per week. Wednesday is a mandatory
in-office day for all teams. Remote days must be logged in the HR portal by
Friday of the previous week.

## Annual Leave
Full-time employees receive 24 days of paid annual leave per calendar year.
A maximum of 5 unused days may be carried over into the next year; the rest
expire on 31 March.

## Expenses
Business expenses must be submitted within 30 days with a receipt. The meal
allowance during business travel is capped at 40 USD per day.

## Equipment
Every employee receives a laptop, refreshed every 3 years. Lost or stolen
devices must be reported to IT within 24 hours.

## Learning Budget
Each employee has an annual learning budget of 1,200 USD for courses,
certifications, and conferences. Unused budget does not roll over.

## Security
Multi-factor authentication is mandatory on all company accounts. Suspected
security incidents must be reported to security@novatech.example within 1 hour.
""".strip()

SALARIES = """
# NovaTech Solutions — Salary Bands (HR only)

## Engineering Salary Bands
Junior engineers earn 70,000 to 90,000 USD. Senior engineers earn 140,000 to
175,000 USD. Bands are reviewed every January.
""".strip()

## 3. Ingestion flow: upload fast, process in the background

Parsing, chunking and embedding a 300-page PDF can take minutes. If `/upload` did that work inside the HTTP request, big files would time out.

So the production app splits it in two:
- **`/upload`** saves the file, creates a database row with `status="pending"`, and returns **immediately** (HTTP 202).
- **A separate worker process** (`app/worker.py`) checks for pending documents every 5 seconds and does the heavy work.

The frontend polls the status until it says `completed`.

In [3]:
STOPWORDS = {"a", "an", "the", "is", "are", "and", "or", "of", "to", "in", "on",
             "at", "for", "per", "what", "how", "many", "much", "does", "do",
             "i", "my", "can", "get", "it", "be", "must", "by", "with"}

def words(text: str) -> set[str]:
    return {w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOPWORDS}


DB = {}      # stand-in for PostgreSQL: one row per document
INDEX = []   # stand-in for ChromaDB + BM25: one entry per chunk


def upload(filename: str, text: str, user_id: int) -> dict:
    """app/api/upload.py: register the document and return at once."""
    doc_id = f"doc-{len(DB) + 1}"
    DB[doc_id] = {"filename": filename, "owner": user_id, "text": text, "status": "pending"}
    return {"document_id": doc_id, "status": "pending"}


def worker_tick():
    """app/worker.py: one polling cycle. Parse → chunk → embed → index."""
    for doc_id, row in DB.items():
        if row["status"] != "pending":
            continue
        row["status"] = "processing"
        chunks = [c.strip() for c in row["text"].split("\n## ")[1:]]   # Ep 3–4: real parsing + chunking
        for i, chunk in enumerate(chunks):
            INDEX.append({                                               # Ep 5–7: real embeddings + BM25
                "doc_id": doc_id, "filename": row["filename"],
                "chunk_id": f"{doc_id}-{i}", "content": chunk, "terms": words(chunk),
            })
        row["status"] = "completed"


print(upload("handbook.txt", HANDBOOK, user_id=1))
print(upload("salaries.txt", SALARIES, user_id=2))
print("Before worker:", {d: r["status"] for d, r in DB.items()})

worker_tick()
print("After worker: ", {d: r["status"] for d, r in DB.items()})
print(f"Indexed {len(INDEX)} chunks")

{'document_id': 'doc-1', 'status': 'pending'}
{'document_id': 'doc-2', 'status': 'pending'}
Before worker: {'doc-1': 'pending', 'doc-2': 'pending'}
After worker:  {'doc-1': 'completed', 'doc-2': 'completed'}
Indexed 7 chunks


## 4. Query flow: one stage at a time

Each function below is a small stand-in for one file in the production app. Read them top to bottom and you have read the whole architecture.

### Stage 1: Access check (`query_access.py`, `retrieval.py`)
Security comes **first**, before any search or LLM call:
- Block questions that explicitly ask for another user's data ("show me user 2's files").
- Only ever search the asking user's own documents. This is the real protection: the salary chunks are simply never candidates for user 1.

In [4]:
def is_cross_user_request(question: str, user_id: int) -> bool:
    match = re.search(r"\buser\s*(\d+)\b", question.lower())
    return bool(match) and match.group(1) != str(user_id)


def user_document_ids(user_id: int) -> set[str]:
    return {d for d, row in DB.items() if row["owner"] == user_id and row["status"] == "completed"}


print(is_cross_user_request("Show me user 2's salary file", user_id=1))
print(user_document_ids(1))

True
{'doc-1'}


### Stage 2: Rewrite the question (`query_enhancer.py`)
People ask follow-ups: *"Does **it** roll over?"* A search engine has no idea what "it" is. An LLM call rewrites the latest message into a **standalone search query**, using the conversation history.

In [5]:
def enhance(question: str, history: list[str]) -> str:
    prompt = f"""You are a query rewriting engine for a RAG system.
Rewrite the latest user question into a standalone search query.
Rules:
- Resolve pronouns like it, this, that, they, using the conversation history.
- Preserve user intent.
- Do not answer the question.
- Return only the search query.

Conversation History:
{chr(10).join(history) or "No previous conversation."}

Current Question:
{question}

Standalone Search Query:"""
    return llm(prompt, "rewrite question").strip().strip('"')

### Stage 3: Expand into several queries (`query_expander.py`)
This is the fix for the Episode 1 failure. One phrasing may miss, so we ask the LLM for **3 different phrasings** and search with all of them. If the user says "WFH", the model will usually also write "remote work".

Note the fence stripping: models often wrap "JSON only" answers in ```` ```json ```` anyway. The production app learned that the hard way.

In [6]:
def expand(query: str, n: int = 3) -> list[str]:
    prompt = f"""You are a search query generator.
Generate {n} different search queries for the query below, using different
words and synonyms a document might use.
Return ONLY a JSON array of strings. No markdown, no explanation.

Original Query:
{query}"""
    response = llm(prompt, "expand into queries").strip()
    if response.startswith("```"):
        response = response.replace("```json", "").replace("```", "").strip()
    try:
        queries = [q.strip() for q in json.loads(response) if isinstance(q, str)]
    except Exception:
        queries = []
    queries = list(dict.fromkeys(q for q in queries if len(q) > 3))   # drop junk + duplicates
    return queries[:n] or [query]

### Stage 4: Retrieve (`retrieval.py`, `hybrid_search.py`)
Run **every** query, but only over the user's own documents. Keep each chunk's best score, and return a shortlist of up to 10.

Our scorer is still Episode 1's word overlap, now scaled to 0–1 (the share of query words found in the chunk). Production combines **vector search (60%) + BM25 keyword search (40%)**. Episodes 5–8 build that.

In [7]:
def retrieve(queries: list[str], doc_ids: set[str], top_k: int = 10) -> list[dict]:
    best = {}
    for query in queries:
        q = words(query)
        for chunk in INDEX:
            if chunk["doc_id"] not in doc_ids or not q:
                continue
            score = len(q & chunk["terms"]) / len(q)
            if score > best.get(chunk["chunk_id"], (0, None))[0]:
                best[chunk["chunk_id"]] = (score, chunk)
    hits = [{**chunk, "score": round(score, 2)} for score, chunk in best.values()]
    return sorted(hits, key=lambda h: h["score"], reverse=True)[:top_k]

### Stage 5: Rerank (`reranker.py`)
Retrieval is fast but rough. The reranker is slow but accurate: a **cross-encoder** reads the question and each chunk *together* and scores the pair. It is too slow to run on every chunk, so it only runs on the shortlist: **top 10 → top 3**, and anything under a minimum score is dropped.

Stand-in here: keep the score order, apply the threshold, keep 3. Episode 10 swaps in the real cross-encoder, and Episode 11 covers the threshold bug the production app shipped with.

In [8]:
def rerank(hits: list[dict], threshold: float = 0.2, top_k: int = 3) -> list[dict]:
    return [h for h in hits if h["score"] >= threshold][:top_k]

### Stage 6: Context management (`context_deduplicator.py`, `context_compressor.py`, `context_window_manager.py`)
Before anything goes into the prompt:
1. **Dedupe:** overlapping chunks can repeat the same text.
2. **Compress:** keep only sentences that mention the question's words. If none match, keep the whole chunk.
3. **Budget:** stop adding chunks once the context would exceed the token budget (production: 4,000 tokens minus 1,000 reserved for the answer).

In [9]:
def manage_context(question: str, hits: list[dict], budget: int = 3000) -> list[dict]:
    unique, seen = [], set()
    for h in hits:                                   # 1. dedupe
        if h["content"] not in seen:
            seen.add(h["content"])
            unique.append(dict(h))

    keywords = [w.lower() for w in re.findall(r"\w+", question) if len(w) > 2]
    for h in unique:                                 # 2. compress
        sentences = re.split(r"(?<=[.!?])\s+", h["content"])
        relevant = [s for s in sentences if any(k in s.lower() for k in keywords)]
        if relevant:
            h["content"] = "\n".join(relevant)

    selected, used = [], 0
    for h in unique:                                 # 3. budget (~1 token per word)
        tokens = len(h["content"].split())
        if used + tokens > budget:
            break
        selected.append(h)
        used += tokens
    return selected

### Stage 7: Prompt + generate (`context_formatter.py`, `prompt_builder.py`, `llm/`)
The same grounding rules as Episode 1, plus conversation history and labeled source blocks.

In [10]:
FALLBACK = "I don't have enough information from the provided documents."

def build_prompt(question: str, sources: list[dict], history: list[str]) -> str:
    context = "\n\n".join(
        f"SOURCE {i + 1} ({s['filename']}, score {s['score']}):\n{s['content']}"
        for i, s in enumerate(sources)
    )
    return f"""You are an assistant that answers questions from provided documents.

Rules:
1. Answer only using the retrieved context.
2. Do not use external knowledge or assume facts.
3. If the answer is not in the context, respond exactly: "{FALLBACK}"
4. Never mention SOURCE labels, filenames or scores. Write natural prose.

Conversation History:
{chr(10).join(history) or "No previous conversation."}

Retrieved Context:
{context or "(no relevant context found)"}

User Question:
{question}

Answer:"""

### Stage 8: Hallucination guard (`hallucination_guard_service.py`)
Don't trust the first answer. A **second LLM call** checks: *is every claim in this answer supported by the context?* If not, regenerate with a stricter prompt.

Let's test the guard on its own with an answer we know is made up:

In [11]:
def is_grounded(question: str, context: str, answer: str) -> bool:
    prompt = f"""You are a hallucination detection system.
Check whether the answer is fully supported by the provided context.
Do not use external knowledge. Return only JSON:
{{"grounded": true or false, "unsupported_claims": []}}

Question: {question}
Context: {context}
Answer: {answer}"""
    response = llm(prompt, "hallucination guard").strip()
    if response.startswith("```"):
        response = response.replace("```json", "").replace("```", "").strip()
    try:
        return bool(json.loads(response)["grounded"])
    except Exception:
        return False   # can't verify → treat as not grounded


learning = next(c["content"] for c in INDEX if c["content"].startswith("Learning Budget"))
print("True answer grounded?", is_grounded("What is the learning budget?", learning, "1,200 USD per year, and it does not roll over."))
print("Fake answer grounded?", is_grounded("What is the learning budget?", learning, "5,000 USD per year, and unused budget rolls over."))

True answer grounded? True


Fake answer grounded? False


### Stage 9: Quality score + sources (`search_evaluator.py`, `source_builder.py`)
The answer is returned with **where it came from** (filename, chunk, score) and a quality badge from the best retrieval score: **Excellent** (≥ 0.7), **Good** (≥ 0.5) or **Weak**. The user can check the source instead of trusting the bot.

In [12]:
def quality(sources: list[dict]) -> str:
    best = max((s["score"] for s in sources), default=0)
    return "Excellent" if best >= 0.7 else "Good" if best >= 0.5 else "Weak"

## 5. Put it together: the orchestrator (`rag_chat.py`)

This is the whole query flow in one function. It prints what happens at each stage, so you can follow a single question through the system.

In [13]:
def chat(question: str, user_id: int, history: list[str] | None = None) -> dict:
    history = history or []
    LLM_CALLS.clear()
    start = time.perf_counter()

    def step(n, name, detail):
        print(f"[{n}] {name:<18} {detail}")

    # 1. Access check: before any search or LLM call
    if is_cross_user_request(question, user_id):
        step(1, "access check", "BLOCKED: asks for another user's data")
        return {"answer": FALLBACK, "sources": [], "llm_calls": 0}
    doc_ids = user_document_ids(user_id)
    step(1, "access check", f"user {user_id} may search {sorted(doc_ids)}")

    # 2–3. Understand the question
    query = enhance(question, history)
    step(2, "rewrite", repr(query))
    queries = expand(query)
    step(3, "expand", queries)

    # 4–5. Find the right chunks
    hits = retrieve(queries, doc_ids)
    step(4, "retrieve", [(h["chunk_id"], h["score"]) for h in hits])
    hits = rerank(hits)
    step(5, "rerank", [(h["chunk_id"], h["score"]) for h in hits])
    if not hits:
        step(6, "no context", "nothing relevant: refuse without calling the LLM again")
        return {"answer": FALLBACK, "sources": [], "llm_calls": len(LLM_CALLS)}

    # 6. Fit the context
    sources = manage_context(query, hits)   # the rewritten query: "it" means nothing here
    step(6, "context", f"{len(sources)} chunk(s), {sum(len(s['content'].split()) for s in sources)} words")

    # 7. Generate
    answer = llm(build_prompt(question, sources, history), "generate answer")
    step(7, "generate", repr(answer[:80]))

    # 8. Check it
    context = "\n\n".join(s["content"] for s in sources)
    if is_grounded(query, context, answer):
        step(8, "guard", "grounded ✓")
    else:
        step(8, "guard", "NOT grounded → regenerating with a strict prompt")
        answer = llm(build_prompt(question, sources, []) + "\nBe strict. If unsure, use the fallback.", "strict regenerate")

    # 9. Attach sources + quality
    result = {
        "answer": answer,
        "sources": [(s["filename"], s["chunk_id"], s["score"]) for s in sources],
        "quality": quality(sources),
        "llm_calls": len(LLM_CALLS),
        "seconds": round(time.perf_counter() - start, 1),
    }
    step(9, "response", f"quality={result['quality']}, {result['llm_calls']} LLM calls, {result['seconds']}s")
    return result

## 6. The Episode 1 failure, again

Same question that broke Episode 1. Watch stage 3.

In [14]:
result = chat("How often am I allowed to WFH?", user_id=1)
print("\nAnswer :", result["answer"])
print("Sources:", result["sources"])

[1] access check       user 1 may search ['doc-1']


[2] rewrite            'WFH policy frequency'


[3] expand             ['Remote work policy adoption rates', 'Work from home policy implementation statistics', 'Frequency of remote work policies in companies']
[4] retrieve           [('doc-1-0', 0.4)]
[5] rerank             [('doc-1-0', 0.4)]
[6] context            1 chunk(s), 36 words


[7] generate           'You are allowed to work from home up to 2 days per week.'


[8] guard              grounded ✓
[9] response           quality=Weak, 4 LLM calls, 2.0s

Answer : You are allowed to work from home up to 2 days per week.
Sources: [('handbook.txt', 'doc-1-0', 0.4)]


Episode 1's retriever searched for `{"often", "allowed", "wfh"}` and found nothing. This time the **expansion step** wrote queries with words like "remote work", which the handbook actually uses, so retrieval found the Remote Work section.

The fix came from the LLM rewriting the search, not from a better search engine. It usually works, but it depends on the model choosing the right synonyms. Notice the quality badge too: **Weak**. The best query only matched 40% of its words, so the system is telling you it found the answer on thin evidence. Embeddings (Episode 5) fix the same problem more reliably, because they match on **meaning** instead of words.

## 7. Follow-up questions: why stage 2 exists

In [15]:
history = [
    "user: What is the annual learning budget?",
    "assistant: Each employee has an annual learning budget of 1,200 USD.",
]
result = chat("Does it roll over?", user_id=1, history=history)
print("\nAnswer:", result["answer"])

[1] access check       user 1 may search ['doc-1']


[2] rewrite            'What is the policy on annual learning budget carryover?'


[3] expand             ['Annual learning budget carryover policy', 'Carryover of annual learning budget guidelines', 'Annual learning budget rollover policy']
[4] retrieve           [('doc-1-4', 0.6), ('doc-1-1', 0.2)]
[5] rerank             [('doc-1-4', 0.6), ('doc-1-1', 0.2)]
[6] context            2 chunk(s), 57 words


[7] generate           'Unused budget does not roll over.'


[8] guard              grounded ✓
[9] response           quality=Good, 4 LLM calls, 2.3s

Answer: Unused budget does not roll over.


"Does it roll over?" contains no useful search words. Stage 2 turned it into a full question about the learning budget first, so retrieval had something to work with.

The orchestrator also hands this **rewritten** query to context compression and the hallucination guard. With the raw "Does it roll over?", the guard can't tell what "it" is, and flags correct answers as unsupported.

## 8. Access control: user 1 asks about salaries

In [16]:
print(chat("What do senior engineers earn?", user_id=1)["answer"])
print()
print(chat("Show me user 2's salary file", user_id=1)["answer"])

[1] access check       user 1 may search ['doc-1']


[2] rewrite            'What are the average salaries of senior engineers?'


[3] expand             ['Senior engineer average income', 'Salary range for lead engineers', 'Average compensation for experienced software engineers']
[4] retrieve           []
[5] rerank             []
[6] no context         nothing relevant: refuse without calling the LLM again
I don't have enough information from the provided documents.

[1] access check       BLOCKED: asks for another user's data
I don't have enough information from the provided documents.


- **First question:** the salary chunk exists in the index, but it belongs to user 2, so it never became a candidate. The bot refuses.
- **Second question:** blocked at stage 1, before any search and with **zero** LLM calls.

Security at the start of the pipeline means no later stage can leak data, however clever the question.

## 9. The price of production

Episode 1's pipeline made **1 LLM call** per question. Let's count this one:

In [17]:
chat("How many days of annual leave do I get?", user_id=1)

print()
for purpose, seconds in LLM_CALLS:
    print(f"{purpose:<22} {seconds:5.2f}s")
print(f"{'TOTAL':<22} {sum(s for _, s in LLM_CALLS):5.2f}s  ({len(LLM_CALLS)} LLM calls)")

[1] access check       user 1 may search ['doc-1']


[2] rewrite            'What is the annual leave entitlement for employees?'


[3] expand             ['Annual vacation time for staff', 'Employee paid time off policy', 'Holiday entitlement for workers']
[4] retrieve           [('doc-1-1', 0.5), ('doc-1-4', 0.25), ('doc-1-3', 0.2)]
[5] rerank             [('doc-1-1', 0.5), ('doc-1-4', 0.25), ('doc-1-3', 0.2)]
[6] context            3 chunk(s), 73 words


[7] generate           'Full-time employees receive 24 days of paid annual leave per calendar year.'


[8] guard              grounded ✓
[9] response           quality=Good, 4 LLM calls, 2.3s

rewrite question        0.28s
expand into queries     0.62s
generate answer         0.75s
hallucination guard     0.63s
TOTAL                   2.28s  (4 LLM calls)


**4 LLM calls** for one answer (5 if the guard rejects the first one). Each stage buys something:

| Stage | Extra LLM call? | What it buys |
|---|---|---|
| Rewrite | ✅ | Follow-up questions work |
| Expand | ✅ | Different wording still finds the answer |
| Rerank | ❌ (small local model) | Only the 3 best chunks reach the LLM |
| Context management | ❌ | Smaller, cheaper, more focused prompt |
| Hallucination guard | ✅ | Fewer confident wrong answers |

Every stage costs time and money, so **each one must earn its place**. Later episodes measure whether it does (Episode 16: evaluating RAG).

## Recap

| | Episode 1 | Episode 2 |
|---|---|---|
| Stages | 4 | 9 |
| LLM calls per question | 1 | 4–5 |
| "WFH" question | ❌ | ✅ (via query expansion) |
| Follow-up questions | ❌ | ✅ |
| Multiple users | ❌ | ✅ scoped to their own documents |
| Checks its own answer | ❌ | ✅ |
| Answer shows its sources | ❌ | ✅ |

**Production RAG = the same 4 steps, with a stage before and after each one to make it safe, accurate and checkable.**

**Next, Episode 3:** real documents aren't neat markdown strings. We parse actual **PDF, DOCX and TXT** files.

⭐ Notebooks: https://github.com/JakkulaMahendhar/rag-from-scratch
🏗️ Full production app: https://github.com/JakkulaMahendhar/rag-chatbot
📸 Instagram: [@codingwithmahi](https://instagram.com/codingwithmahi)